# 5. Logistic Regression

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')


In [6]:
# --- Load and preprocess churn dataset (Classification) ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_churn = pd.read_csv('churn_df.csv')
# Dropping ID columns which have no predictive value
df_churn = df_churn.drop(columns=['RowNumber', 'CustomerId', 'Surname'], errors='ignore')

X_clf = df_churn.drop(columns=['Exited'])
y_clf = df_churn['Exited']

# Handling missing values
X_clf.fillna(X_clf.mean(numeric_only=True), inplace=True)
for col in X_clf.select_dtypes(include=['object']).columns:
    X_clf[col].fillna(X_clf[col].mode()[0], inplace=True)

# Categorical mapping using get_dummies
X_clf = pd.get_dummies(X_clf, drop_first=True)

# Train Test Split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# Scaling
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)
print("Churn dataset ready for classification!")


Churn dataset ready for classification!


## Scikit-Learn Logistic Regression
Optimization: Added `class_weight='balanced'` for vastly improved minority class recall.

In [7]:
from sklearn.linear_model import LogisticRegression

log_clf = LogisticRegression(class_weight='balanced', random_state=42)
log_clf.fit(X_train_clf_scaled, y_train_clf)
log_preds = log_clf.predict(X_test_clf_scaled)

print("Accuracy:", accuracy_score(y_test_clf, log_preds))
print(classification_report(y_test_clf, log_preds))


Accuracy: 0.7195
              precision    recall  f1-score   support

           0       0.91      0.72      0.81      1607
           1       0.38      0.71      0.50       393

    accuracy                           0.72      2000
   macro avg       0.65      0.72      0.65      2000
weighted avg       0.81      0.72      0.75      2000



## Logistic Regression from scratch (Gradient Descent) - Predict Adjust
Manually adjusted prediction threshold to 0.3 to increase Class 1 recall naturally.

In [8]:
class MyLogisticRegression:
    def __init__(self, lr=0.01, epochs=1000):
        self.lr = lr
        self.epochs = epochs

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0
        
        for _ in range(self.epochs):
            linear_model = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(linear_model)
            
            dw = (1/m) * np.dot(X.T, (y_pred - y))
            db = (1/m) * np.sum(y_pred - y)
            
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_model)

my_log_clf = MyLogisticRegression(lr=0.1, epochs=500)
my_log_clf.fit(X_train_clf_scaled, y_train_clf.values)

# Lowering threshold from standard 0.5 to 0.3 to improve recall significantly
my_log_probs = my_log_clf.predict_proba(X_test_clf_scaled)
my_log_preds = (my_log_probs >= 0.3).astype(int)

print("Scratch Accuracy:", accuracy_score(y_test_clf, my_log_preds))
print(classification_report(y_test_clf, my_log_preds))


Scratch Accuracy: 0.7955
              precision    recall  f1-score   support

           0       0.88      0.86      0.87      1607
           1       0.48      0.53      0.50       393

    accuracy                           0.80      2000
   macro avg       0.68      0.69      0.69      2000
weighted avg       0.80      0.80      0.80      2000

